# University Chapters — Azure Medallion Pipeline (Synapse Spark)

End-to-end **Bronze → Silver → Gold** run on Azure Synapse Spark (runtime 3.5, Delta Lake 3.2),
adapted from the local `pipeline/` package in this repository — same DQ rules, same ordering,
same fail-loud policy, with cloud-grade upgrades:

* all four layers live on **ADLS Gen2** (`abfss://`) as **Delta tables** (parquet for raw Bronze),
* Gold is published with a single **atomic `MERGE`** on `chapter_id` (update + insert +
  delete-vanished) — the production idempotency choice promised in the README,
* the run summary is appended to a `_runs` Delta table for auditability.

**Parameters** (set by the Synapse pipeline, or manually when running interactively):
`storage_account` (required), `lake_container` (default `lake`), `source` (`live` | `fixture`),
`run_id` (optional override). DQ rules: **DQ-Q1** quarantines invalid/missing coordinates
(`INVALID_COORDINATES`); **DQ-W1** publishes missing/blank/`UNKNOWN` city with
`dq_status='WARNING'` (`MISSING_OR_UNKNOWN_CITY`). Empty OR/WA is a data fact; an empty batch
or CA=0 is a hard failure.

In [ ]:
# Parameters cell (Synapse injects pipeline values here)
storage_account = ""          # required, e.g. "stunichapters001"
lake_container = "lake"       # ADLS Gen2 filesystem holding the medallion layers
source = "live"               # "live" (public FeatureServer) or "fixture" (seeded DQ rows)
run_id = ""                   # optional explicit run id (normally left empty)

In [ ]:
"""Configuration, constants and helpers. Mirrors pipeline/config.py + dq_rules.py."""
import json
import logging
from datetime import datetime, timezone

import requests
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

try:  # Synapse runtime provides notebookutils; keep a shim hook for local testing
    from notebookutils import mssparkutils  # type: ignore
except ImportError:
    mssparkutils  # noqa: B018  (injected by the local test harness)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("university_chapters")

if not storage_account:
    raise ValueError("Parameter 'storage_account' is required (ADLS Gen2 account name).")

FEATURE_SERVER_URL = (
    "https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/"
    "UniversityChapters_Public/FeatureServer/0"
)
QUERY_URL = f"{FEATURE_SERVER_URL}/query"
WHERE_CLAUSE = "State IN ('CA','OR','WA')"
IN_SCOPE_STATES = ("CA", "OR", "WA")
PAGE_SIZE = 1000
HTTP_TIMEOUT_SECONDS = 30

LAKE = f"abfss://{lake_container}@{storage_account}.dfs.core.windows.net"
DATASET = "university_chapters"
BRONZE_ROOT = f"{LAKE}/bronze/{DATASET}"
SILVER_PATH = f"{LAKE}/silver/{DATASET}"
GOLD_PATH = f"{LAKE}/gold/{DATASET}/v1"
QUARANTINE_PATH = f"{LAKE}/quarantine/{DATASET}"
RUNS_PATH = f"{LAKE}/_runs/{DATASET}"
FIXTURE_PATH = f"{LAKE}/fixtures/university_chapters_fixture.json"

REASON_INVALID_COORDINATES = "INVALID_COORDINATES"
REASON_MISSING_OR_UNKNOWN_CITY = "MISSING_OR_UNKNOWN_CITY"
DQ_STATUS_OK, DQ_STATUS_WARNING = "OK", "WARNING"
LON_MIN, LON_MAX, LAT_MIN, LAT_MAX = -180.0, 180.0, -90.0, 90.0

GOLD_COLUMNS = [
    "chapter_id", "chapter_name", "city", "state", "longitude", "latitude",
    "dq_status", "dq_warnings", "ingest_run_id", "ingested_at_utc",
]


class PipelineError(RuntimeError):
    """Any raise in this notebook fails the Synapse activity -> pipeline run fails loudly."""

In [ ]:
"""Bronze ingest — land the payload exactly as received, plus metadata. No transforms."""


def new_run_id(now=None):
    now = now or datetime.now(timezone.utc)
    return now.strftime("run_%Y%m%dT%H%M%SZ")


def _query_page(session, offset):
    params = {
        "where": WHERE_CLAUSE, "outFields": "*", "returnGeometry": "true",
        "f": "json", "resultOffset": offset, "resultRecordCount": PAGE_SIZE,
    }
    resp = session.get(QUERY_URL, params=params, timeout=HTTP_TIMEOUT_SECONDS)
    if resp.status_code != 200:
        raise PipelineError(f"FeatureServer returned HTTP {resp.status_code}: {resp.text[:500]}")
    payload = resp.json()
    if "error" in payload:  # ArcGIS reports many failures inside a 200 response
        raise PipelineError(f"FeatureServer returned an error payload: {payload['error']}")
    if "features" not in payload:
        raise PipelineError(f"Unexpected response shape: {list(payload.keys())}")
    return payload


def fetch_live_pages():
    pages, offset = [], 0
    with requests.Session() as session:
        while True:
            payload = _query_page(session, offset)
            pages.append(payload)
            n = len(payload["features"])
            log.info("Fetched page %d: %d features (offset=%d)", len(pages), n, offset)
            if payload.get("exceededTransferLimit") and n > 0:
                offset += n
            else:
                return pages


def load_fixture_pages():
    rows = spark.read.text(FIXTURE_PATH, wholetext=True).collect()
    if not rows:
        raise PipelineError(f"Fixture not found or empty: {FIXTURE_PATH}")
    return [json.loads(rows[0]["value"])]


def ingest_bronze(source, run_id=None):
    started_at = datetime.now(timezone.utc)
    run_id = run_id or new_run_id(started_at)
    ingest_date = started_at.strftime("%Y-%m-%d")

    if source == "live":
        pages, source_detail = fetch_live_pages(), QUERY_URL
    elif source == "fixture":
        pages, source_detail = load_fixture_pages(), FIXTURE_PATH
    else:
        raise PipelineError(f"Unknown source '{source}' (expected 'live' or 'fixture')")

    rows_in = sum(len(p["features"]) for p in pages)
    if rows_in == 0:
        raise PipelineError(
            "FeatureServer returned 0 features for CA/OR/WA — refusing to land an empty "
            "bronze batch (empty OR/WA is a data fact, an empty batch is not)."
        )

    bronze_dir = f"{BRONZE_ROOT}/ingest_date={ingest_date}/run_id={run_id}"
    for i, payload in enumerate(pages, start=1):
        mssparkutils.fs.put(f"{bronze_dir}/page_{i:04d}.json", json.dumps(payload), True)
    metadata = {
        "run_id": run_id, "ingest_date": ingest_date,
        "ingested_at_utc": started_at.isoformat(), "source": source,
        "source_detail": source_detail, "where_clause": WHERE_CLAUSE,
        "pages": len(pages), "rows_in": rows_in,
    }
    mssparkutils.fs.put(f"{bronze_dir}/_ingest_metadata.json", json.dumps(metadata, indent=2), True)
    log.info("Bronze landed: %s (%d rows, %d page(s))", bronze_dir, rows_in, len(pages))
    return run_id, bronze_dir, metadata

In [ ]:
"""DQ predicates — kept in exact parity with pipeline/dq_rules.py in this repo."""


def invalid_coordinates(lon, lat):
    """DQ-Q1: missing, null, non-numeric (null after try_cast), NaN or out-of-range."""
    lon_bad = lon.isNull() | F.isnan(lon) | (lon < F.lit(LON_MIN)) | (lon > F.lit(LON_MAX))
    lat_bad = lat.isNull() | F.isnan(lat) | (lat < F.lit(LAT_MIN)) | (lat > F.lit(LAT_MAX))
    return lon_bad | lat_bad


def missing_or_unknown_city(city):
    """DQ-W1: null, blank/whitespace, or literal 'UNKNOWN' (case-insensitive)."""
    trimmed = F.trim(city)
    return city.isNull() | (trimmed == F.lit("")) | (F.upper(trimmed) == F.lit("UNKNOWN"))


def with_dq_flags(df):
    quarantined = invalid_coordinates(F.col("longitude"), F.col("latitude"))
    warned = missing_or_unknown_city(F.col("city"))
    return (
        df.withColumn("is_quarantined", quarantined)
        .withColumn("quarantine_reason", F.when(quarantined, F.lit(REASON_INVALID_COORDINATES)))
        .withColumn("dq_status",
                    F.when(warned, F.lit(DQ_STATUS_WARNING)).otherwise(F.lit(DQ_STATUS_OK)))
        .withColumn("dq_warnings",
                    F.when(warned, F.array(F.lit(REASON_MISSING_OR_UNKNOWN_CITY)))
                    .otherwise(F.array().cast("array<string>")))
    )

In [ ]:
"""Silver transform — flatten, type, scope-filter, DQ-Q1 quarantine, dedupe, DQ-W1 flag.
Quarantine and Silver are written as Delta tables."""


def build_silver(run_id, bronze_dir, metadata):
    raw = spark.read.option("multiLine", "true").json(f"{bronze_dir}/page_*.json")
    if "features" not in raw.columns:
        raise PipelineError(f"Bronze payload at {bronze_dir} has no 'features' array")

    flat = (
        raw.select(F.explode("features").alias("feature"))
        .select(
            F.trim(F.col("feature.attributes.ChapterID").cast("string")).alias("chapter_id"),
            F.trim(F.col("feature.attributes.University_Chapter").cast("string")).alias("chapter_name"),
            F.trim(F.col("feature.attributes.City").cast("string")).alias("city"),
            F.upper(F.trim(F.col("feature.attributes.State").cast("string"))).alias("state"),
            # try_cast: malformed coordinates must become NULL (-> DQ-Q1), not crash the job
            F.expr("try_cast(feature.geometry.x as double)").alias("longitude"),
            F.expr("try_cast(feature.geometry.y as double)").alias("latitude"),
            F.col("feature.attributes.OBJECTID").cast("long").alias("source_object_id"),
            F.to_json(F.col("feature")).alias("raw_payload"),
        )
        .withColumn("ingest_run_id", F.lit(metadata["run_id"]))
        .withColumn("ingested_at_utc", F.lit(metadata["ingested_at_utc"]).cast("timestamp"))
    )

    rows_in = flat.count()
    scoped = flat.filter(F.col("state").isin(*IN_SCOPE_STATES))
    rows_out_of_scope = rows_in - scoped.count()

    flagged = with_dq_flags(scoped)

    quarantine_df = flagged.filter(F.col("is_quarantined")).select(
        "chapter_id", "chapter_name", "city", "state", "longitude", "latitude",
        "source_object_id", "quarantine_reason", "ingest_run_id", "ingested_at_utc",
        "raw_payload",
    )
    survivors = flagged.filter(~F.col("is_quarantined"))

    # Dedupe AFTER quarantine (a bad duplicate can never shadow a good record).
    w = Window.partitionBy("chapter_id").orderBy(
        F.col("source_object_id").desc_nulls_last(), F.col("chapter_name").asc()
    )
    deduped = survivors.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")

    silver_df = deduped.select(
        "chapter_id", "chapter_name", "city", "state", "longitude", "latitude",
        "dq_status", "dq_warnings", "source_object_id", "ingest_run_id", "ingested_at_utc",
    )

    counts = {
        "rows_in": rows_in,
        "rows_out_of_scope": rows_out_of_scope,
        "rows_quarantined": quarantine_df.count(),
        "rows_deduped": survivors.count() - deduped.count(),
        "rows_warned": silver_df.filter(F.col("dq_status") == DQ_STATUS_WARNING).count(),
        "rows_ok": silver_df.filter(F.col("dq_status") == DQ_STATUS_OK).count(),
    }

    # Quarantine: Delta, append-by-run, partitioned by run for easy inspection.
    (quarantine_df.write.format("delta").mode("append")
     .partitionBy("ingest_run_id").save(QUARANTINE_PATH))
    # Silver: Delta, full overwrite each run (latest snapshot; history via Delta versions).
    (silver_df.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").save(SILVER_PATH))

    log.info("Silver written: %s", counts)
    return counts

In [ ]:
"""Gold publish — gates, then one atomic MERGE (upsert + delete-vanished) on chapter_id."""


def publish_gold():
    silver = spark.read.format("delta").load(SILVER_PATH)

    rows_silver = silver.count()
    if rows_silver == 0:
        raise PipelineError("Silver is empty — refusing to publish an empty Gold snapshot.")
    bad_status = silver.filter(~F.col("dq_status").isin(DQ_STATUS_OK, DQ_STATUS_WARNING)).count()
    if bad_status:
        raise PipelineError(f"{bad_status} row(s) with unexpected dq_status — aborting publish.")
    if silver.filter(F.col("state") == "CA").count() == 0:
        raise PipelineError(
            "CA row count is 0 but CA has a non-zero baseline — refusing to publish. "
            "(Empty OR/WA alone is an expected source data fact and does not block.)"
        )

    gold_src = silver.select(*GOLD_COLUMNS)
    if DeltaTable.isDeltaTable(spark, GOLD_PATH):
        (DeltaTable.forPath(spark, GOLD_PATH).alias("tgt")
         .merge(gold_src.alias("src"), "tgt.chapter_id = src.chapter_id")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .whenNotMatchedBySourceDelete()   # full-snapshot source: vanished chapters retire
         .execute())
        log.info("Gold MERGE completed (upsert + delete-vanished on chapter_id).")
    else:
        gold_src.write.format("delta").save(GOLD_PATH)
        log.info("Gold created at first publish: %s", GOLD_PATH)

    gold = spark.read.format("delta").load(GOLD_PATH)
    by_state = {r["state"]: r["n"] for r in gold.groupBy("state").agg(F.count("*").alias("n")).collect()}
    return {
        "rows_gold": gold.count(),
        "rows_gold_ca": by_state.get("CA", 0),
        "rows_gold_or": by_state.get("OR", 0),
        "rows_gold_wa": by_state.get("WA", 0),
    }

In [ ]:
"""Execute the full run and append the summary to the _runs Delta table."""
started = datetime.now(timezone.utc)

effective_run_id, bronze_dir, metadata = ingest_bronze(source, run_id or None)
silver_counts = build_silver(effective_run_id, bronze_dir, metadata)
gold_counts = publish_gold()

summary = {
    "run_id": effective_run_id, "source": source,
    "started_at_utc": started.isoformat(),
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "bronze_dir": bronze_dir, **silver_counts, **gold_counts,
}
(spark.createDataFrame([summary]).write.format("delta")
 .option("mergeSchema", "true").mode("append").save(RUNS_PATH))

log.info(
    "RUN SUMMARY %s | rows_in=%d rows_quarantined=%d rows_warned=%d rows_ok=%d "
    "rows_gold=%d (CA=%d OR=%d WA=%d)",
    summary["run_id"], summary["rows_in"], summary["rows_quarantined"],
    summary["rows_warned"], summary["rows_ok"], summary["rows_gold"],
    summary["rows_gold_ca"], summary["rows_gold_or"], summary["rows_gold_wa"],
)
print(json.dumps(summary, indent=2))

## Verifying the product

**Spark SQL** (attach any notebook to the pool):
```sql
SELECT * FROM delta.`abfss://lake@<storage_account>.dfs.core.windows.net/gold/university_chapters/v1`
ORDER BY chapter_id;
DESCRIBE HISTORY delta.`abfss://lake@<storage_account>.dfs.core.windows.net/gold/university_chapters/v1`;
```

**Serverless SQL pool** (no Spark needed, pay-per-query):
```sql
SELECT TOP 100 *
FROM OPENROWSET(
    BULK 'https://<storage_account>.dfs.core.windows.net/lake/gold/university_chapters/v1',
    FORMAT = 'DELTA'
) AS gold;
```

Delta gives time travel for free: `SELECT ... FROM delta.` + `VERSION AS OF n` reproduces any
earlier published snapshot — the cloud upgrade of the local "overwrite loses history" trade-off.